In [2]:
# ==========================================
# КОНФИГУРАЦИЯ И ПУТИ ПРОЕКТА
# ==========================================
import os
import numpy as np
import pandas as pd

USB_ROOT = "/media/melting/5A80-1912/student_data"
DIR_1_3 = os.path.join(USB_ROOT, "1_3")
DIR_1_4 = os.path.join(USB_ROOT, "1_4")
DIR_1_5 = os.path.join(USB_ROOT, "1_5")

os.makedirs(DIR_1_5, exist_ok=True)

NOSOLOGIES_FILE = os.path.join(DIR_1_3, "nosologies.json")
INPUT_MEASUREMENTS = os.path.join(DIR_1_4, "measurements_with_legend.npz")
IMPUTED_MATRIX = os.path.join(DIR_1_5, "mrt_matrix_knn_imputed.npz")

In [3]:
from sklearn.impute import KNNImputer

print("--- Загрузка предварительно очищенных данных из папки 1_5 ---")
# Загружаем текущий файл из папки 1_5 (где уже удалены столбцы с >5% пропусков)
data = np.load(IMPUTED_MATRIX, allow_pickle=True)
X = data["data"]
cols = data["cols"]
rows = data["rows"]

X_df = pd.DataFrame(X, columns=cols, index=rows)

# Проверяем наличие пропусков (NaN или нулей)
print(f"🔍 Количество пропусков (NaN) в матрице: {X_df.isna().sum().sum()}")
print(f"🔍 Количество нулей в матрице: (X_df == 0).sum().sum()")

# Если есть нули, которые нужно интерпретировать как пропуски для импутации:
X_df_nan = X_df.replace(0, np.nan)

# Запускаем KNNImputer для оставшихся единичных пропусков
print("--- Запуск алгоритма KNNImputer для 'зеленых' признаков ---")
imputer = KNNImputer(n_neighbors=5)
X_imputed_array = imputer.fit_transform(X_df_nan)

# Собираем обратно в DataFrame
X_imputed_df = pd.DataFrame(X_imputed_array, columns=cols, index=rows)

print(f"✅ Пропусков после импутации осталось: {X_imputed_df.isna().sum().sum()}")

# Перезаписываем итоговый файл в папке 1_5
np.savez_compressed(
    IMPUTED_MATRIX,
    data=X_imputed_df.values,
    cols=cols,
    rows=rows
)
print(f"💾 Готово! Финальная матрица успешно обновлена и сохранена в: {IMPUTED_MATRIX}")

--- Загрузка предварительно очищенных данных из папки 1_5 ---
🔍 Количество пропусков (NaN) в матрице: 0
🔍 Количество нулей в матрице: (X_df == 0).sum().sum()
--- Запуск алгоритма KNNImputer для 'зеленых' признаков ---
✅ Пропусков после импутации осталось: 0
💾 Готово! Финальная матрица успешно обновлена и сохранена в: /media/melting/5A80-1912/student_data/1_5/mrt_matrix_knn_imputed.npz


In [4]:
import numpy as np
import pandas as pd

# Загружаем текущий файл из папки 1_5 (путь уже настроен в конфигурации)
data = np.load(IMPUTED_MATRIX, allow_pickle=True)
X = data["data"]
cols = data["cols"]
rows = data["rows"]

X_df = pd.DataFrame(X, columns=cols, index=rows)

# Считаем нули и NaN
total_cells = X_df.size
nan_count = X_df.isna().sum().sum()
zero_count = (X_df == 0).sum().sum()

print("--- АНАЛИЗ ДАННЫХ В ПАПКЕ 1_5 ---")
print(f"📦 Всего ячеек в матрице: {total_cells:,}")
print(f"❌ Количество пропусков (NaN): {nan_count}")
print(f"0️⃣ Количество нулей (0): {zero_count}")

# Если нулей или пропусков много, можно посмотреть по столбцам:
if zero_count > 0 or nan_count > 0:
    print("\n⚠️ Столбцы, в которых остались нули:")
    zeros_per_col = (X_df == 0).sum()
    print(zeros_per_col[zeros_per_col > 0])
else:
    print("\n✅ Нулей и пропусков в данных больше нет! Матрица полностью чистая.")

--- АНАЛИЗ ДАННЫХ В ПАПКЕ 1_5 ---
📦 Всего ячеек в матрице: 2,277,045
❌ Количество пропусков (NaN): 0
0️⃣ Количество нулей (0): 0

✅ Нулей и пропусков в данных больше нет! Матрица полностью чистая.


In [5]:
import json

# Открываем файл и смотрим ключи для первого попавшегося пациента
with open(NOSOLOGIES_FILE, 'r', encoding='utf-8') as f:
    nosologies_data = json.load(f)

first_id = list(nosologies_data.keys())[0]
print(f"Ключи в словаре для пациента {first_id}:")
print(list(nosologies_data[first_id].keys()))

Ключи в словаре для пациента sub-NDARINVAG023WG3:
['hypothalamic_subunits_volumes.v', 'lh.aparc.pial', 'rh.aparc', 'wmparc', 'lh.w-g.pct', 'hipposubfields.rh.T1.v22', 'rh.BA_exvivo.thresh', 'rh.aparc.a2009s', 'vsinus', 'thalamic-nuclei.lh.v13.T1', 'lh.aparc', 'thalamic-nuclei.rh.v13.T1', 'lh.BA_exvivo', 'rh.aparc.pial', 'lh.BA_exvivo.thresh', 'lh.aparc.a2009s', 'rh.aparc.DKTatlas', 'hipposubfields.lh.T1.v22', 'rh.w-g.pct', 'amygdalar-nuclei.lh.T1.v22', 'amygdalar-nuclei.rh.T1.v22', 'entowm', 'lh.aparc.DKTatlas', 'rh.BA_exvivo', 'aseg', 'BRAINVOL', 'tcp_demos', 'dataset', 'nosology', 'gender', 'age', 'valid']


In [17]:
import numpy as np
import pandas as pd

print("--- Генерация детального аудита со средними значениями ---")

# 1. Загружаем матрицу через общую константу
data_load = np.load(IMPUTED_MATRIX, allow_pickle=True)
X = data_load["data"]
cols = data_load["cols"]
rows = data_load["rows"]

X_df = pd.DataFrame(X, columns=cols, index=rows)

# 2. Загружаем список аномальных пациентов
iqr_df = pd.read_csv('/media/melting/5A80-1912/student_data/1_5/iqr_patients_with_more_than_50_outliers.csv')
anomaly_pids = iqr_df['patient_id'].tolist()

# 3. Считаем статистические границы IQR и средние/стандартные отклонения по всей выборке
Q1 = X_df.quantile(0.25)
Q3 = X_df.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

pop_mean = X_df.mean()
pop_std = X_df.std()

detailed_records = []

for pid in anomaly_pids:
    if pid not in X_df.index:
        continue
    
    patient_row = X_df.loc[pid]
    outlier_mask = (patient_row < lower_bound) | (patient_row > upper_bound)
    bad_features = patient_row[outlier_mask].index.tolist()
    
    for feat in bad_features:
        val = patient_row[feat]
        mean_val = pop_mean[feat]
        std_val = pop_std[feat]
        
        # Считаем Z-score (насколько отклонение выражено в стандартных отклонениях)
        z_score = (val - mean_val) / std_val if std_val > 0 else 0
        
        if val == 0:
            status = "0️⃣ Нулевой баг"
        elif val < lower_bound[feat]:
            status = "📉 Ниже нормы"
        else:
            status = "📈 Выше нормы"
            
        detailed_records.append({
            'patient_id': pid,
            'feature_name': feat,
            'patient_value': round(val, 4),
            'population_mean': round(mean_val, 4),
            'z_score': round(z_score, 2),
            'anomaly_type': status
        })

# Превращаем в датафрейм и сохраняем в новый CSV
df_detailed = pd.DataFrame(detailed_records)
report_filename = "anomalies_with_population_means.csv"
df_detailed.to_csv(report_filename, index=False, encoding='utf-8-sig')

print(f"📊 Отчет успешно создан и сохранен в файл: {report_filename}")
print(f"🔍 Всего проанализировано аномальных точек: {len(df_detailed)}")

--- Генерация детального аудита со средними значениями ---
📊 Отчет успешно создан и сохранен в файл: anomalies_with_population_means.csv
🔍 Всего проанализировано аномальных точек: 1930


In [29]:
import numpy as np
import pandas as pd

print("--- Пересчет Z-score на очищенной популяционной базе ---")

# 1. Загружаем матрицу через глобальную константу
data_load = np.load(IMPUTED_MATRIX, allow_pickle=True)
X = data_load["data"]
cols = data_load["cols"]
rows = data_load["rows"]
X_df = pd.DataFrame(X, columns=cols, index=rows)

# 2. Загружаем список первичных аномалий от IQR
iqr_df = pd.read_csv('/media/melting/5A80-1912/student_data/1_5/iqr_patients_with_more_than_50_outliers.csv')

# Список пациентов с колоссальным числом выбросов (>200), которые точно портят статистику нормы
# (например, sub-10159 и sub-10269)
severe_outliers = iqr_df[iqr_df['broken_features_count'] > 200]['patient_id'].tolist()
print(f"🧹 Исключаем из расчета нормы заведомо бракованных пациентов: {severe_outliers}")

# 3. Создаем «очищенную» выборку для расчета эталонной нормы ($\mu$ и $\sigma$)
X_clean_baseline = X_df.drop(index=[pid for pid in severe_outliers if pid in X_df.index])

# Считаем честные средние и стандартные отклонения по чистой группе
clean_mean = X_clean_baseline.mean()
clean_std = X_clean_baseline.std().replace(0, np.nan)

# 4. Рассчитываем честный Z-score для ВСЕХ пациентов относительно чистой базы
Z_clean_df = (X_df - clean_mean) / clean_std

# 5. Соберим заново детальный отчет с честными Z-оценками для подозрительных пациентов
anomaly_pids = iqr_df['patient_id'].tolist()
detailed_records = []

# Пороговые границы для IQR тоже пересчитаем по чистой базе
Q1_clean = X_clean_baseline.quantile(0.25)
Q3_clean = X_clean_baseline.quantile(0.75)
IQR_clean = Q3_clean - Q1_clean
lower_b = Q1_clean - 3 * IQR_clean
upper_b = Q3_clean + 3 * IQR_clean

for pid in anomaly_pids:
    if pid not in X_df.index:
        continue
    
    patient_row = X_df.loc[pid]
    patient_z = Z_clean_df.loc[pid]
    
    # Находим признаки, вылетевшие за границы
    outlier_mask = (patient_row < lower_b) | (patient_row > upper_b)
    bad_features = patient_row[outlier_mask].index.tolist()
    
    for feat in bad_features:
        val = patient_row[feat]
        z_val = patient_z[feat]
        
        if val == 0:
            status = "0️⃣ Нулевой баг"
        elif val < lower_b[feat]:
            status = "📉 Ниже нормы"
        else:
            status = "📈 Выше нормы"
            
        detailed_records.append({
            'patient_id': pid,
            'feature_name': feat,
            'patient_value': round(val, 4),
            'clean_population_mean': round(clean_mean[feat], 4),
            'true_z_score': round(z_val, 2),
            'anomaly_type': status
        })

# Сохраняем обновленный честный отчет
df_true_z = pd.DataFrame(detailed_records)
new_report_filename = "anomalies_with_clean_population_means.csv"
df_true_z.to_csv(new_report_filename, index=False, encoding='utf-8-sig')

print(f"✅ Честный отчет с пересчитанными Z-score сохранен в файл: {new_report_filename}")

# Посмотрим топ-5 самых сильных отклонений по честному Z-score
display(df_true_z.sort_values(by='true_z_score', key=abs, ascending=False).head(10))

--- Пересчет Z-score на очищенной популяционной базе ---
🧹 Исключаем из расчета нормы заведомо бракованных пациентов: ['sub-10159', 'sub-10269', 'sub-11062', 'sub-50022']
✅ Честный отчет с пересчитанными Z-score сохранен в файл: anomalies_with_clean_population_means.csv


,patient_id,feature_name,patient_value,clean_population_mean,true_z_score,anomaly_type
36,sub-NDARINVBL733HBP,rh.aparc.pial/insula/FoldInd,61427.000000,234.651901,22.24,📈 Выше нормы
33,sub-NDARINVBL733HBP,rh.aparc.pial/medialorbitofrontal/FoldInd,491.000000,62.480900,12.66,📈 Выше нормы
189,sub-10159,rh.aparc.a2009s/G_and_S_paracentral/CurvInd,7.000000,1.668000,11.22,📈 Выше нормы
183,sub-10159,rh.w-g.pct/parsopercularis/Min,-94.427498,-0.061500,-11.18,📉 Ниже нормы
101,sub-NDARINVKZ112BTB,lh.BA_exvivo.thresh/MT_exvivo.thresh/GausCurv,0.086000,0.024500,10.67,📈 Выше нормы
37,sub-NDARINVBL733HBP,wmparc/wm-lh-pericalcarine/normMax,170.000000,113.388298,10.50,📈 Выше нормы
1667,sub-50022,lh.aparc.DKTatlas/parsopercularis/CurvInd,6.900000,2.079700,10.43,📈 Выше нормы
1457,sub-50022,lh.aparc.a2009s/G_front_inf-Opercular/CurvInd,5.100000,1.494800,10.18,📈 Выше нормы
1674,sub-50022,lh.aparc/parsopercularis/CurvInd,7.200000,2.208500,10.06,📈 Выше нормы
19,sub-NDARINVBL733HBP,wmparc/wm-rh-supramarginal/normStdDev,14.348600,7.789200,10.03,📈 Выше нормы


In [30]:
import os
import json
import numpy as np
import pandas as pd

print("--- Сборка обновленного мастер-отчета с честными Z-score ---")

# 1. Пути к файлам отчетов
IQR_FILE = '/media/melting/5A80-1912/student_data/1_5/iqr_patients_with_more_than_50_outliers.csv'
ISO_FILE = '/media/melting/5A80-1912/student_data/1_5/isolation_forest_anomalies_list.csv'
CLEAN_Z_REPORT = '/media/melting/5A80-1912/student_data/1_5/anomalies_with_clean_population_means.csv' # Новый файл с честными Z-оценками

# 2. Загружаем базовые списки
iqr_df = pd.read_csv(IQR_FILE)
iso_df = pd.read_csv(ISO_FILE)

iqr_set = set(iqr_df['patient_id'])
iso_set = set(iso_df['patient_id'])
all_anomalies = sorted(list(iqr_set.union(iso_set)))

# 3. Загружаем nosologies.json через ваш глобальный конфиг
with open(NOSOLOGIES_FILE, 'r', encoding='utf-8') as f:
    nosologies_data = json.load(f)

# 4. Загружаем честный отчет с true_z_score и считаем количество жестких выбросов (|Z| > 3)
if os.path.exists(CLEAN_Z_REPORT):
    df_clean_z = pd.read_csv(CLEAN_Z_REPORT)
    df_clean_z['true_z_score'] = pd.to_numeric(df_clean_z['true_z_score'], errors='coerce')
    # Считаем количество признаков с |Z| > 3 для каждого пациента по чистой базе
    extreme_counts_clean = df_clean_z[df_clean_z['true_z_score'].abs() > 3].groupby('patient_id').size().to_dict()
else:
    extreme_counts_clean = {}
    print("⚠️ Файл с честными z_score не найден! Сначала запустите ячейку пересчета нормы.")

# 5. Собираем финальную мастер-таблицу
results = []
for pid in all_anomalies:
    patient_info = nosologies_data.get(pid, {})
    
    dataset = patient_info.get("dataset", "N/A")
    nosology = patient_info.get("nosology", "N/A")
    age = patient_info.get("age", "N/A")
    gender = patient_info.get("gender", "N/A")
    
    iqr_row = iqr_df[iqr_df['patient_id'] == pid]
    broken_count = iqr_row['broken_features_count'].values[0] if not iqr_row.empty else 0
    is_iso = pid in iso_set
    
    # Берем количество экстремальных выбросов по честной базе
    z5_clean_count = extreme_counts_clean.get(pid, 0)

    results.append({
        'patient_id': pid,
        'IQR выбросов': broken_count,
        'Isolation Forest': '🚨 Да' if is_iso else 'Нет',
        'Экстремальных (|Z|>3, чистая база)': z5_clean_count,
        'Датасет': dataset,
        'Нозология / Диагноз': nosology
    })

df_master_clean = pd.DataFrame(results)

# Сортируем по числу честных экстремальных выбросов по убыванию
df_master_clean = df_master_clean.sort_values(by='Экстремальных (|Z|>3, чистая база)', ascending=False)

# Сохраняем в итоговый CSV-файл
master_filename = "master_anomalies_audit_clean.csv"
df_master_clean.to_csv(master_filename, index=False, encoding='utf-8-sig')

print(f"✅ Обновленный мастер-отчет успешно сохранен в файл: {master_filename}")

# Выводим красивую таблицу в блокнот
display(df_master_clean)

--- Сборка обновленного мастер-отчета с честными Z-score ---
✅ Обновленный мастер-отчет успешно сохранен в файл: master_anomalies_audit_clean.csv


,patient_id,IQR выбросов,Isolation Forest,"Экстремальных (|Z|>3, чистая база)",Датасет,Нозология / Диагноз
0,sub-10159,356,🚨 Да,352,UCLA,CONTROL
1,sub-10269,225,🚨 Да,239,UCLA,CONTROL
4,sub-11062,230,🚨 Да,227,UCLA,CONTROL
7,sub-50022,217,🚨 Да,219,UCLA,SCHZ
9,sub-50075,132,Нет,136,UCLA,SCHZ
6,sub-50006,134,🚨 Да,135,UCLA,SCHZ
5,sub-11098,108,🚨 Да,108,UCLA,CONTROL
13,sub-NDARINVKZ112BTB,110,🚨 Да,103,TCP,OTHER
11,sub-70021,91,🚨 Да,90,UCLA,ADHD
3,sub-10707,84,🚨 Да,85,UCLA,CONTROL


In [31]:
# Настраиваем pandas, чтобы он показывал все колонки без сжатия
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Сортируем и сбрасываем индексы, чтобы они шли по порядку от 0 до N
df_master_clean = df_master_clean.sort_values(
    by='Экстремальных (|Z|>3, чистая база)', 
    ascending=False
).reset_index(drop=True)

# Выводим красивую и полную таблицу
display(df_master_clean)

,patient_id,IQR выбросов,Isolation Forest,"Экстремальных (|Z|>3, чистая база)",Датасет,Нозология / Диагноз
0,sub-10159,356,🚨 Да,352,UCLA,CONTROL
1,sub-10269,225,🚨 Да,239,UCLA,CONTROL
2,sub-11062,230,🚨 Да,227,UCLA,CONTROL
3,sub-50022,217,🚨 Да,219,UCLA,SCHZ
4,sub-50075,132,Нет,136,UCLA,SCHZ
5,sub-50006,134,🚨 Да,135,UCLA,SCHZ
6,sub-11098,108,🚨 Да,108,UCLA,CONTROL
7,sub-NDARINVKZ112BTB,110,🚨 Да,103,TCP,OTHER
8,sub-70021,91,🚨 Да,90,UCLA,ADHD
9,sub-10707,84,🚨 Да,85,UCLA,CONTROL


In [35]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import diptest
except ImportError:
    !pip install diptest
    import diptest

print("--- Надежный поиск бимодальности с очисткой от грубых выбросов ---")

# 1. Загружаем матрицу
data_load = np.load(IMPUTED_MATRIX, allow_pickle=True)
X = data_load["data"]
cols = data_load["cols"]
rows = data_load["rows"]
X_df = pd.DataFrame(X, columns=cols, index=rows)

# 2. Функция расчета коэффициента Сарле
def calculate_sarles_bc(series):
    clean_series = series.dropna()
    n = len(clean_series)
    if n < 4:
        return 0.0
    g = clean_series.skew()
    k = clean_series.kurtosis()
    numerator = g**2 + 1
    denominator = k + (3 * (n - 1)**2) / ((n - 2) * (n - 3))
    if denominator == 0:
        return 0.0
    return numerator / denominator

results = []

for col in X_df.columns:
    vals = X_df[col].dropna()
    if len(vals) < 10:
        continue
    
    # 🧹 Фильтруем грубые артефакты (выбросы за пределами 3 IQR) для честного анализа формы
    q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
    iqr = q3 - q1
    filtered_vals = vals[(vals >= q1 - 3*iqr) & (vals <= q3 + 3*iqr)]
    
    if len(filtered_vals) < 10:
        continue
        
    bc = calculate_sarles_bc(filtered_vals)
    
    try:
        dip_stat, p_value = diptest.diptest(filtered_vals.values)
    except Exception:
        p_value = 1.0
    
    # 🔑 СОВМЕЩЕННЫЙ КРИТЕРИЙ: Сарле > 0.55 И дип-тест значим (p < 0.05)
    if bc > (5/9) and p_value < 0.05:
        results.append({
            'feature_name': col,
            'sarle_bc': round(bc, 4),
            'dip_p_value': round(p_value, 4)
        })

df_bimodal = pd.DataFrame(results)

if not df_bimodal.empty:
    # Сортируем по степени бимодальности (Сарле)
    df_bimodal = df_bimodal.sort_values(by='sarle_bc', ascending=False).reset_index(drop=True)
    top_50_df = df_bimodal.head(50)
    
    report_filename = "bimodal_features_audit_report.csv"
    df_bimodal.to_csv(report_filename, index=False, encoding='utf-8-sig')
    
    USB_ROOT = "/media/melting/5A80-1912/student_data"
    output_dir = os.path.join(USB_ROOT, "5e")
    os.makedirs(output_dir, exist_ok=True)
    
    # Перестраиваем графики с ограничением выбросов по оси X для красивого вида
    for idx, row in top_50_df.iterrows():
        feat = row['feature_name']
        bc = row['sarle_bc']
        pval = row['dip_p_value']
        
        raw_vals = X_df[feat].dropna()
        q1, q3 = raw_vals.quantile(0.25), raw_vals.quantile(0.75)
        iqr = q3 - q1
        plot_vals = raw_vals[(raw_vals >= q1 - 3*iqr) & (raw_vals <= q3 + 3*iqr)]
        
        plt.figure(figsize=(9, 5))
        sns.kdeplot(plot_vals, fill=True, color='teal', alpha=0.4)
        
        plt.title(f"№{idx+1}: {feat}\nSarle BC: {bc} | Dip p-val: {pval}", fontsize=9)
        plt.xlabel("Значение признака (без крайних выбросов)", fontsize=10)
        plt.ylabel("Плотность (KDE)", fontsize=10)
        plt.grid(True, linestyle='--', alpha=0.5)
        plt.tight_layout()
        
        safe_feat_name = "".join(c if c.isalnum() or c in ('_', '-') else '_' for c in str(feat))
        filename = os.path.join(output_dir, f"bimodal_{idx+1:02d}_{safe_feat_name[:40]}.png")
        
        plt.savefig(filename, dpi=300)
        plt.close()
        
    print(f"✅ Успешно отфильтровано и сохранено честных бимодальных графиков: {len(top_50_df)}")
    display(top_50_df.head(10))
else:
    print("✅ По строгим совмещенным критериям бимодальных признаков не найдено (или все выбросы были артефактами).")

--- Надежный поиск бимодальности с очисткой от грубых выбросов ---
✅ Успешно отфильтровано и сохранено честных бимодальных графиков: 50


,feature_name,sarle_bc,dip_p_value
0,rh.aparc.a2009s/Lat_Fis-ant-Vertical/FoldInd,0.6763,0.0000
1,rh.aparc.pial/transversetemporal/FoldInd,0.6705,0.0000
2,lh.w-g.pct/precuneus/Range,0.6617,0.0371
3,wmparc/wm-lh-superiorfrontal/normRange,0.6469,0.0004
4,lh.aparc.pial/transversetemporal/FoldInd,0.6445,0.0000
5,wmparc/wm-rh-inferiorparietal/normMin,0.6383,0.0000
6,wmparc/wm-lh-cuneus/normMin,0.6347,0.0000
7,aseg/Right-Caudate/normMin,0.6333,0.0001
8,wmparc/wm-lh-inferiorparietal/normMin,0.6324,0.0000
9,lh.w-g.pct/inferiorparietal/Mean,0.6237,0.0000
